# Bird Radar Trajectory Visualization

Interactive 3D visualization and feature analysis for bird radar tracks.

This notebook demonstrates:
- 3D trajectory plotting by species
- Kinematic feature distributions
- Flight pattern analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from shapely import wkb
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Load Data

In [ ]:
# Load training data
df = pd.read_csv('../data/train.csv')
print(f"Loaded {len(df)} training samples")
print(f"\nBird species distribution:")
print(df['bird_group'].value_counts())

## 3D Trajectory Visualization

In [ ]:
def parse_trajectory(wkb_hex):
    """Parse WKB trajectory to coordinates."""
    try:
        geom = wkb.loads(wkb_hex, hex=True)
        return np.array(list(geom.coords))
    except:
        return None

def plot_trajectory_3d(coords, title="Bird Trajectory", color='blue'):
    """Plot single 3D trajectory."""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], 
            color=color, linewidth=2, marker='o', markersize=3)
    
    # Mark start and end
    ax.scatter(coords[0, 0], coords[0, 1], coords[0, 2], 
               color='green', s=100, marker='^', label='Start')
    ax.scatter(coords[-1, 0], coords[-1, 1], coords[-1, 2], 
               color='red', s=100, marker='v', label='End')
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_zlabel('Altitude (m)')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    return fig

In [ ]:
# Plot sample trajectories for each species
species_list = df['bird_group'].unique()[:6]  # First 6 species

for species in species_list:
    sample = df[df['bird_group'] == species].iloc[0]
    coords = parse_trajectory(sample['trajectory'])
    
    if coords is not None and len(coords) > 1:
        plot_trajectory_3d(coords, title=f"{species} - Track {sample['track_id']}")
        plt.show()

## Multi-Species Comparison

In [ ]:
# Compare trajectories across species
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')

colors = plt.cm.tab10(np.linspace(0, 1, len(species_list)))

for i, species in enumerate(species_list):
    species_df = df[df['bird_group'] == species]
    sample = species_df.iloc[0]
    coords = parse_trajectory(sample['trajectory'])
    
    if coords is not None and len(coords) > 1:
        ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], 
                color=colors[i], linewidth=2, label=species, alpha=0.7)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Altitude (m)')
ax.set_title('Bird Flight Trajectories by Species')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Kinematic Feature Analysis

In [ ]:
# Extract features for analysis
from src.features import extract_all_features

print("Extracting features from sample data...")
sample_df = df.sample(n=min(200, len(df)), random_state=42)
features = extract_all_features(sample_df)

# Merge with labels
features_with_labels = features.merge(
    sample_df[['track_id', 'bird_group']], 
    on='track_id'
)

print(f"Extracted {len(features.columns)} features for {len(features)} samples")

In [ ]:
# Velocity distribution by species
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Horizontal velocity
features_with_labels.boxplot(column='horizontal_velocity_mean', 
                             by='bird_group', ax=axes[0, 0])
axes[0, 0].set_title('Horizontal Velocity by Species')
axes[0, 0].set_ylabel('Velocity (m/s)')

# Vertical velocity (climb rate)
features_with_labels.boxplot(column='climb_rate_mean', 
                             by='bird_group', ax=axes[0, 1])
axes[0, 1].set_title('Climb Rate by Species')
axes[0, 1].set_ylabel('Climb Rate (m/s)')

# Acceleration variance (flight pattern indicator)
features_with_labels.boxplot(column='acceleration_variance', 
                             by='bird_group', ax=axes[1, 0])
axes[1, 0].set_title('Acceleration Variance by Species')
axes[1, 0].set_ylabel('Variance')

# Path tortuosity
features_with_labels.boxplot(column='tortuosity', 
                             by='bird_group', ax=axes[1, 1])
axes[1, 1].set_title('Path Tortuosity by Species')
axes[1, 1].set_ylabel('Tortuosity Ratio')

plt.suptitle('Kinematic Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# Flight pattern classification (Flapping vs Gliding)
fig, ax = plt.subplots(figsize=(12, 6))

flight_patterns = features_with_labels.groupby('bird_group')[['is_flapping', 'is_gliding']].mean()
flight_patterns.plot(kind='bar', ax=ax, color=['#ff6b6b', '#4ecdc4'])

ax.set_title('Flight Pattern Distribution by Species', fontsize=14)
ax.set_ylabel('Proportion')
ax.set_xlabel('Bird Species')
ax.legend(['Flapping', 'Gliding'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Feature Correlation Heatmap

In [ ]:
# Select kinematic features for correlation analysis
kinematic_features = [
    'velocity_mean', 'velocity_max', 'horizontal_velocity_mean',
    'vertical_velocity_mean', 'acceleration_mean', 'acceleration_variance',
    'climb_rate_mean', 'descent_rate_max', 'turn_radius_mean',
    'path_curvature', 'tortuosity', 'traj_straightness'
]

# Filter to available columns
available_features = [f for f in kinematic_features if f in features.columns]

# Compute correlation matrix
corr_matrix = features[available_features].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Kinematic Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## Altitude Profile Analysis

In [ ]:
# Altitude distribution by species
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Mean altitude
altitude_stats = features_with_labels.groupby('bird_group')['traj_z_mean'].agg(['mean', 'std'])
altitude_stats['mean'].plot(kind='bar', ax=axes[0], color='steelblue', 
                            yerr=altitude_stats['std'], capsize=5)
axes[0].set_title('Average Flight Altitude by Species')
axes[0].set_ylabel('Altitude (m)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Altitude range
features_with_labels.boxplot(column='altitude_range', by='bird_group', ax=axes[1])
axes[1].set_title('Altitude Range by Species')
axes[1].set_ylabel('Range (m)')

plt.suptitle('Altitude Profile Analysis')
plt.tight_layout()
plt.show()

## Key Insights

From the visualizations above:

1. **Flight Patterns**: Different species show distinct kinematic signatures
   - Birds of Prey: Gliding flight (low acceleration variance)
   - Songbirds: Flapping flight (high acceleration variance)

2. **Altitude Preferences**: Species occupy different vertical niches
   - High-altitude: Birds of Prey
   - Mid-altitude: Gulls, Geese
   - Low-altitude: Songbirds

3. **Path Characteristics**:
   - Straight fliers: Geese, Ducks (high straightness, low tortuosity)
   - Maneuverable fliers: Songbirds (low straightness, high tortuosity)

These kinematic features provide physics-based discrimination between species.